In [3]:
import torch
import transformers
import pandas as pd
import numpy as np
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Environment ready!")

PyTorch: 2.13.0+cpu
Transformers: 4.46.3
Pandas: 3.0.3
NumPy: 2.5.1
Environment ready!


# DocuTune: Parameter-Efficient Text Classification

## Objective

This project investigates whether parameter-efficient fine-tuning
using LoRA can improve classification performance while reducing
the number of trainable parameters compared with full fine-tuning.

## Initial Task

Classify company documents into three categories:

- Invoices
- Purchase Orders
- Shipping Orders

The dataset contains 2,469 labeled documents with a relatively balanced
distribution across the three document categories.

In [4]:
from sklearn.datasets import fetch_20newsgroups

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.med",
    "talk.politics.misc"
]

train_data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

test_data = fetch_20newsgroups(
    subset="test",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

print("Training samples:", len(train_data.data))
print("Testing samples:", len(test_data.data))
print("Categories:", train_data.target_names)

KeyboardInterrupt: 

In [5]:
from pathlib import Path

data_path = Path("../data/raw/train-00000-of-00001.parquet")

print("File exists:", data_path.exists())
print("File path:", data_path)

File exists: True
File path: ..\data\raw\train-00000-of-00001.parquet


In [6]:
df = pd.read_parquet(
    "../data/raw/train-00000-of-00001.parquet",
    engine="fastparquet"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (2469, 5)

Columns:
['file_content', 'file_name', 'extracted_data', 'document_type', 'chat_format']


In [7]:
print("Document types:")
print(df["document_type"].value_counts())

print("\nUnique document types:")
print(df["document_type"].unique())

Document types:
document_type
Purchase Orders    830
Invoices           830
Shipping Orders    809
Name: count, dtype: int64

Unique document types:
['Shipping Orders' 'Purchase Orders' 'Invoices']


In [8]:
pd.set_option("display.max_colwidth", 1000)

sample = df.iloc[0]

print("FILE NAME:")
print(sample["file_name"])

print("\nDOCUMENT TYPE:")
print(sample["document_type"])

print("\nFILE CONTENT:")
print(sample["file_content"])

print("\nEXTRACTED DATA:")
print(sample["extracted_data"])

FILE NAME:
order_10562.pdf

DOCUMENT TYPE:
Shipping Orders

FILE CONTENT:
Order ID: 10562
Shipping Details:
Ship Name: Reggiani Caseifici
Ship Address: Strada Provinciale 124
Ship City: Reggio Emilia
Ship Region: Southern Europe
Ship Postal Code: 42100
Ship Country: Italy
Customer Details:
Customer ID: REGGC
Customer Name: Reggiani Caseifici
Employee Details:
Employee Name: Nancy Davolio
Shipper Details:
Shipper ID: 1
Shipper Name: Speedy Express
Order Details:
Order Date: 2017-06-09
Shipped Date: 2017-06-12
Products:
--------------------------------------------------------------------------------------------------
Product: Geitost
Quantity: 20
Unit Price: 2.5
Total: 50.0
--------------------------------------------------------------------------------------------------
Product: Tarte au sucre
Quantity: 10
Unit Price: 49.3
Total: 493.0
Total Price:
Total Price: 543.0


EXTRACTED DATA:
 ```json
{
  "OrderID": "10562",
  "ShippingDetails": {
    "ShipName": "Reggiani Caseifici",
    "Ship

In [9]:
df["text_length"] = df["file_content"].str.len()
df["word_count"] = df["file_content"].str.split().str.len()

print(df[["text_length", "word_count"]].describe())

       text_length   word_count
count  2469.000000  2469.000000
mean    500.981774    63.366140
std     315.010015    26.953296
min     153.000000    25.000000
25%     232.000000    39.000000
50%     386.000000    59.000000
75%     801.000000    88.000000
max    1493.000000   185.000000


In [12]:
from sklearn.model_selection import train_test_split

# Keep only the columns needed for our ML task
data = df[["file_content", "document_type"]].copy()

# Remove any missing values
data = data.dropna()

print("Total samples:", len(data))
print(data.head())

Total samples: 2469
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [13]:
# Encode document labels as integers
label_mapping = {
    "Invoices": 0,
    "Purchase Orders": 1,
    "Shipping Orders": 2
}

data["label"] = data["document_type"].map(label_mapping)

print(data["label"].value_counts())
print(data.head())

label
1    830
0    830
2    809
Name: count, dtype: int64
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [14]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    stratify=data["label"],
    random_state=42
)

# Second split: divide temporary 30% into 15% validation and 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["document_type"].value_counts())

print("\nValidation distribution:")
print(val_df["document_type"].value_counts())

print("\nTesting distribution:")
print(test_df["document_type"].value_counts())

Training samples: 1728
Validation samples: 370
Testing samples: 371

Training distribution:
document_type
Invoices           581
Purchase Orders    581
Shipping Orders    566
Name: count, dtype: int64

Validation distribution:
document_type
Invoices           124
Purchase Orders    124
Shipping Orders    122
Name: count, dtype: int64

Testing distribution:
document_type
Invoices           125
Purchase Orders    125
Shipping Orders    121
Name: count, dtype: int64


In [16]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "validation.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [17]:
print(list(processed_dir.iterdir()))

[WindowsPath('../data/processed/test.csv'), WindowsPath('../data/processed/train.csv'), WindowsPath('../data/processed/validation.csv')]
